In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "insomnia_dataset" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
dataset_98580683_fitbit_sleep_daily_summary_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (436962, 42538607, 37110488, 4102985, 439013)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_sleep_daily_summary_98580683_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_sleep_daily_summary_98580683",
  "fitbit_sleep_daily_summary_98580683_*.csv")
message(str_glue('The data will be written to {fitbit_sleep_daily_summary_98580683_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_98580683_fitbit_sleep_daily_summary_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_sleep_daily_summary_98580683_path,
  destination_format = "CSV")


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_98580683/fitbit_sleep_daily_summary_98580683_*.csv. Use this path when reading the data into your notebooks in the future.



In [2]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_sleep_daily_summary_98580683_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_sleep <- read_bq_export_from_workspace_bucket(fitbit_sleep_daily_summary_98580683_path)

dim(insomnia_sleep)

head(insomnia_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_98580683/fitbit_sleep_daily_summary_98580683_000000000000.csv.



[1] 2971101      12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2656915,2017-07-23,false,180,172,0,8,NA,31,114,27,8
2669155,2023-03-10,true,180,166,0,14,NA,0,155,11,14
2632811,2020-02-09,false,180,164,7,16,NA,5,153,6,16
2669155,2021-08-18,true,180,162,25,18,NA,20,130,12,18
2632811,2020-12-02,true,180,161,10,19,NA,34,118,9,19


In [3]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded



In [4]:
colnames(insomnia_sleep)[2] <- 'date'
insomnia_sleep_main <- insomnia_sleep |>
  filter(is_main_sleep == "true")

insomnia_sleep_nap <- insomnia_sleep |>
  filter(is_main_sleep == "false")

insomnia_sleep_sum <- sleep_sum_main_nap_func(df = insomnia_sleep)

In [9]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Insomnia/sleep/', df = insomnia_sleep_sum)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  58.0   1st Qu.:  91.0  
 Median : 124.0   Median : 387.0  
 Mean   : 499.8   Mean   : 959.2  
 3rd Qu.: 721.2   3rd Qu.:1696.2  
 Max.   :4166.0   Max.   :4898.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000181   Min.   :   1.0   Min.   :2009-11-18   Min.   :2013-02-17  
 1st Qu.:1811614   1st Qu.:  58.0   1st Qu.:2017-12-22   1st Qu.:2023-06-11  
 Median :2949307   Median : 124.0   Median :2021-11-30   Median :2023-09-27  
 Mean   :3951130   Mean   : 499.8   Mean   :2020-08-05   Mean   :2023-03-21  
 3rd Qu.:5946627   3rd Qu.: 721.2   3rd Qu.:2023-06-17   3rd Qu.:2023-10-01  
 Max.   :9993734   Max.   :4166.0   Max.   :2023-09-30   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :   1.0   Min.   :   0.0   Min.   : 0.000   High (>90%):1290  
 1st Qu.:  91.0   1st Qu.:  17.0   1st Qu.: 9.898   80-90%     : 686  
 Median : 387.0   Median :  73.0   Median :31.432   70-80%     : 500  
 Mean   : 959.2   Mean   : 459.5   Mean   :38.512   60-70%     : 455  
 3rd Qu.:1696.2   3rd Qu.: 611.2   3rd Qu.:65.806   50-60%     : 348  
 Max.   :4898.0   Max.   :39

NULL

In [6]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Insomnia',
    df = insomnia_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Insomnia',
    df = insomnia_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  5084 365.   83.3  378.  323.  419.  201.  475. 
2 minute_awake   5084  50.2  18.8   50.1  40.1  59.4  20.8  76.9
3 minute_deep    4949  56.7  17.0   56.2  45.3  68.2  29.4  84.8
4 minute_in_bed  5084 416.   93.9  432.  370.  476.  230.  538. 
5 minute_light   4949 256.   46.6  254.  224.  284.  185.  331. 
6 minute_rem     4949  75.4  22.8   76.2  60.2  90.8  37.1 110. 
7 minute_wake    4949  55.6  12.1   55.1  47.7  62.9  37.2  76.1
# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  4786 104.  28.6  102.    87.3 118.   65    147. 
2 minute_awake   4786  13.0 10.1    9.63   6.8  15.1   3.67  35.2
3 minute_deep    3503  24.5 11.6   24     17.4  31.2   6     43  
4 minute_in_bed  4786 119.  29.6  115.   101   13

In [7]:
head(insomnia_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = insomnia_sleep_sum, path.out =  '/home/jupyter/workspaces/result/Insomnia/', cohort = 'Insomnia', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000181,2021-02-22,TRUE,FALSE,1,571,544,27,0,103,⋯,27,0,571,544,27,103,248,193,27,95.27145
1000181,2021-02-23,TRUE,TRUE,2,700,633,67,4,96,⋯,63,0,700,633,67,96,295,134,63,90.42857
1000181,2021-02-24,TRUE,FALSE,1,295,274,21,0,47,⋯,21,0,295,274,21,47,120,107,21,92.88136
1000181,2021-02-25,TRUE,FALSE,1,591,533,58,0,99,⋯,58,0,591,533,58,99,308,126,58,90.18613
1000181,2021-02-26,TRUE,TRUE,2,609,537,72,6,69,⋯,64,0,609,537,72,69,224,157,64,88.17734
1000181,2021-02-27,TRUE,FALSE,1,724,682,42,0,129,⋯,42,5,724,682,42,129,322,231,42,94.19890


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3     P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep     5084 381.  82.5   394.  344.  432.  218.   490. 
2 minute_awake      5084  52.2 19.6    51.9  41.7  61.9  22.2   80.2
3 minute_deep       5084  43.9 21.6    44.8  29.4  59.2   4.54  77.3
4 minute_in_bed     5084 435.  93.2   449.  394.  492.  249.   555. 
5 minute_light      5084 197.  82.4   211.  150.  255.   26.4  308. 
6 minute_rem        5084  58.3 28.5    60.3  39.3  78.7   5.72 103. 
7 minute_wake       5084  42.9 18.8    44.8  31.8  56.0   5.50  70.4
8 sleep_efficiency  5084  87.9  4.40   88.4  86.6  90.0  81.2   93.1


In [8]:
miss_gap_func(df = insomnia_sleep_sum, path.out =  '/home/jupyter/workspaces/result/Insomnia/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0     5.0    25.0   199.7   159.0  3409.0 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       4882         96.0        202         3.97
2         7       3656         71.9       1428        28.1 
3        14       2978         58.6       2106        41.4 
4        30       2372         46.7       2712        53.3 
5        60       1806         35.5       3278        64.5 


NULL